# Ch 15　Workflows vs Agents：常見編排模式

> **本 notebook 完全不需要 API key**——我們用「假 LLM 節點」把每種模式的**結構**搭出來跑，重點是看清拓樸與執行路徑。把假節點換成真的 LLM 呼叫，就是線上版。

In [ ]:
!uv pip install -q langgraph

## 15.2　Prompt chaining：線性接力

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    topic: str
    outline: str
    draft: str
    final: str

# 每一步處理上一步的輸出（這裡用假實作；真實情境每個都是一次 LLM 呼叫）
def generate_outline(s): return {"outline": f"《{s['topic']}》大綱：1. 起 2. 承 3. 轉 4. 合"}
def write_draft(s):      return {"draft": f"根據大綱寫成的初稿（{s['outline'][:10]}…）"}
def polish(s):           return {"final": s["draft"] + "（已潤稿）"}

chain = (StateGraph(State)
         .add_node("generate_outline", generate_outline).add_node("write_draft", write_draft).add_node("polish", polish)
         .add_edge(START, "generate_outline").add_edge("generate_outline", "write_draft")
         .add_edge("write_draft", "polish").add_edge("polish", END).compile())
print(chain.invoke({"topic": "LangGraph 入門", "outline": "", "draft": "", "final": ""})["final"])

## 15.3　Parallelization：三路平行 + 彙整（順便量延遲）

從 START 同時指向三個節點 = 同一 super-step 平行跑（Ch7）。平行寫同一欄位要用可合併的 reducer。

In [ ]:
import time
from typing import Annotated
from operator import add

class PState(TypedDict):
    text: str
    findings: Annotated[list[str], add]   # 三個平行節點各寫一筆，用 add 合併（Ch7 的坑）

def check_grammar(s): time.sleep(1); return {"findings": ["文法：OK"]}
def check_tone(s):    time.sleep(1); return {"findings": ["語氣：偏正式"]}
def check_length(s):  time.sleep(1); return {"findings": ["長度：適中"]}
def aggregate(s):     return {"text": s["text"] + " | " + "；".join(s["findings"])}

pg = StateGraph(PState)
for n, f in [("grammar", check_grammar), ("tone", check_tone), ("length", check_length), ("aggregate", aggregate)]:
    pg.add_node(n, f)
for n in ["grammar", "tone", "length"]:
    pg.add_edge(START, n)            # 三條都從 START 出發 = 平行
    pg.add_edge(n, "aggregate")      # 三路收斂到彙整
pg.add_edge("aggregate", END)
graph = pg.compile()

t = time.time()
out = graph.invoke({"text": "一段文字", "findings": []})
print(out["text"])
print(f"三個各 sleep 1 秒，平行總耗時約 {time.time()-t:.1f} 秒（而非 3 秒）—— 這就是平行化的價值")

## 15.4　Routing：先分類，再走專門路徑（Ch5 已見）

In [ ]:
class RState(TypedDict):
    q: str
    answer: str

def classify(s):
    return {}    # 分類結果由 route 直接看 q 判斷；這裡不改 state
def faq(s):     return {"answer": "[FAQ] 這題有現成解答。"}
def human(s):   return {"answer": "[轉真人] 這題較複雜，已轉專員。"}
def route(s):   return "human" if any(k in s["q"] for k in ["退款", "投訴", "壞"]) else "faq"

rg = (StateGraph(RState).add_node("classify", classify).add_node("faq", faq).add_node("human", human)
      .add_edge(START, "classify")
      .add_conditional_edges("classify", route, {"faq": "faq", "human": "human"})
      .add_edge("faq", END).add_edge("human", END).compile())
print(rg.invoke({"q": "怎麼重設密碼？", "answer": ""})["answer"])
print(rg.invoke({"q": "我要退款！", "answer": ""})["answer"])

## 15.6　Evaluator-optimizer：生成 → 評估 → 不合格就回頭（記得設上限）

In [ ]:
class EState(TypedDict):
    score: int
    rounds: int

def generate(s):
    # 假裝每輪都「進步一點」：分數隨輪數上升
    r = s["rounds"] + 1
    return {"rounds": r, "score": 40 + r * 25}

def evaluate(s):
    return {}   # 評估結果由 check 看 score 判斷

def check(s):
    if s["score"] >= 80:          # 達標
        return "done"
    if s["rounds"] >= 5:          # 上限護欄：避免無限打磨（呼應 Ch1 的 max_turns）
        return "done"
    return "retry"

eg = (StateGraph(EState).add_node("generate", generate).add_node("evaluate", evaluate)
      .add_edge(START, "generate").add_edge("generate", "evaluate")
      .add_conditional_edges("evaluate", check, {"retry": "generate", "done": END}).compile())
result = eg.invoke({"score": 0, "rounds": 0})
print(f"跑了 {result['rounds']} 輪，最終分數 {result['score']}（>=80 或達上限才停）")

## 重點回顧

模式由可控到彈性：chaining → routing → parallelization → orchestrator-worker → evaluator-optimizer → 完整 agent。**能用 workflow 就別硬上 agent**；能畫固定流程就畫，真不能才交給模型自決。把上面的假節點換成 LLM 呼叫，即是線上版。